In [ ]:
!pip install sentencepiece transformers tqdm
!git clone https://github.com/ggarciabaez/VLA.git
!pip install --upgrade torch torchvision

In [2]:
import os, shutil
from google.colab import drive
drive.mount("/content/drive")

os.makedirs("/content/data")
os.makedirs("/content/model")
shutil.copytree("/content/VLA/model", "/content/model", dirs_exist_ok=True)
shutil.rmtree("/content/VLA")

shutil.copytree("/content/drive/MyDrive/VLA/mt50/mt50", "/content/data", dirs_exist_ok=True)

Mounted at /content/drive


'/content/data'

In [1]:
import json
import math
import os
import random
import time
from collections import OrderedDict
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.amp import GradScaler
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm
from transformers import SiglipTokenizer

from model.vla import VLA, print_model_counts
from model.utils import VLAConfig

In [2]:
# =========================
# Colab Config (edit here)
# =========================
DATA_DIR = "/content/data/"
CHECKPOINT_DIR = "/content/data/checkpoints"
PROMPTS_JSON = "task_prompts.json"
EPISODE_GLOB = "ep*.npz"
MAX_EPISODE_FILES = None  # int or None

SEED = 42
VAL_SPLIT = 0.1
BATCH_SIZE = 64
NUM_WORKERS = 4
PIN_MEMORY = True
PERSISTENT_WORKERS = False

EPOCHS = 10
LEARNING_RATE = 3e-4
BACKBONE_LR_SCALE = 0.1
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 2
GRAD_CLIP_NORM = 1.0
LOG_EVERY_STEPS = 100
RESUME = False
COMPILE_MODEL = False

# Model config
MODEL_KWARGS = dict(
    n_trainable=4,
    d_model=768,
    n_heads=6,
    n_layers=8,
    latent_size=64,
    action_heads=6,
    action_layers=4,
    chunk_size=32,
    flow_steps=10,
    dropout=0.1,
)

In [3]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_task_name(x) -> str:
    if isinstance(x, bytes):
        return x.decode("utf-8")
    return str(x)

In [4]:
class VLAEpisodeDatasetCache(Dataset):
    """
    Indexes every timestep across nested shard files:
      data_dir/epXXXX/<task>.npz
    Loads lazily — only the file needed for a given index is read from disk.

    Normalization is applied here so that the DataLoader workers handle it
    in parallel rather than on the GPU.
    """

    def __init__(
        self,
        data_dir:  str,
        tokenizer: SiglipTokenizer,
        cfg:       VLAConfig,
        cache_files: int = 2,
    ):
        self.data_dir = Path(data_dir)
        self.tokenizer = tokenizer
        self.cfg      = cfg
        self.cache_files = max(1, int(cache_files))

        # ── normalisation stats ───────────────────────────────────────────────
        stats = np.load(self.data_dir / "norm_stats.npz")
        self.action_mean = torch.from_numpy(stats["action_mean"])
        self.action_std  = torch.from_numpy(stats["action_std"])

        # ── pre-tokenize all prompts ──────────────────────────────────────────
        # task_prompts.json: { task_name: [prompt_a, prompt_b, prompt_c] }
        # Result: { task_name: (3, seq_len) int64 tensor }
        with open(self.data_dir / "task_prompts.json") as f:
            raw_prompts: dict[str, list[str]] = json.load(f)

        self.tokens: dict[str, torch.Tensor] = {}
        for task_name, prompts in raw_prompts.items():
            task_name = normalize_task_name(task_name)
            enc = tokenizer(
                prompts,                  # list of 3 strings → batched
                padding="max_length",     # pads to tokenizer.model_max_length
                truncation=True,
                return_tensors="pt",
            )
            self.tokens[task_name] = enc.input_ids   # (3, seq_len)

        # ── flat index over nested shards ─────────────────────────────────────
        episode_dirs = sorted(p for p in self.data_dir.glob("ep*") if p.is_dir())
        if not episode_dirs:
            raise RuntimeError(
                f"No episode folders found in {data_dir}. "
                "Expected layout: data_dir/epXXXX/<task>.npz"
            )

        shard_paths: list[Path] = []
        for ep_dir in episode_dirs:
            shard_paths.extend(sorted(ep_dir.glob("*.npz")))

        if not shard_paths:
            raise RuntimeError(
                f"No nested shard files found in episode folders under {data_dir}"
            )

        # Keep path/task lookup small (one entry per shard file, not per sample)
        self._shard_paths: list[str] = [str(p) for p in shard_paths]
        self._task_name_by_shard: list[str] = []

        # Build index as compact arrays in C-managed memory.
        shard_idx_list, t_list = [], []
        for i, path in enumerate(shard_paths):
            with np.load(path, mmap_mode="r") as ep:
                T = ep["actions"].shape[0]
                if "task_name" in ep:
                    task_name = normalize_task_name(np.array(ep["task_name"]).reshape(-1)[0])
                else:
                    task_name = path.stem
            self._task_name_by_shard.append(task_name)
            for t in range(T):
                shard_idx_list.append(i)
                t_list.append(t)

        self._idx_shard = np.array(shard_idx_list, dtype=np.int32)
        self._idx_t = np.array(t_list, dtype=np.int32)

        self._fallback_cache: dict[str, torch.Tensor] = {}
        print(
            f"Dataset: {len(episode_dirs)} episode folders, "
            f"{len(shard_paths)} shard files, {len(self._idx_shard):,} samples"
        )
        self._cache: dict[str, np.lib.npyio.NpzFile] = {}
        self._cache_order: list[str] = []

    def _load(self, path: str):
        if path in self._cache:
            self._cache_order.remove(path)
            self._cache_order.append(path)
            return self._cache[path]

        # NpzFile with mmap: array data lives in the OS page cache,
        # not the Python heap. All workers share the same physical pages.
        # Don't dict() it — that would force-load everything eagerly.
        ep = np.load(path, mmap_mode="r")
        self._cache[path] = ep
        self._cache_order.append(path)

        if len(self._cache_order) > self.cache_files:
            old = self._cache_order.pop(0)
            old_ep = self._cache.pop(old, None)
            if old_ep is not None:
                old_ep.close()

        return ep

    def __del__(self):
        for ep in self._cache.values():
            ep.close()
        self._cache.clear()
        self._cache_order.clear()

    def __len__(self) -> int:
        return len(self._idx_shard)

    def __getitem__(self, idx: int) -> dict:
        shard_i = int(self._idx_shard[idx])
        t = int(self._idx_t[idx])
        path = self._shard_paths[shard_i]
        ep = self._load(path)

        img = ep["images"][t]
        if img.ndim == 3 and img.shape[-1] == 3:
            img = np.transpose(img, (2, 0, 1))
        img = torch.from_numpy(img)

        state = torch.from_numpy(ep["states"][t].astype(np.float32))

        ci = ep["chunk_indices"][t]
        ci = ci[ci >= 0]
        ci = np.clip(ci, 0, ep["actions"].shape[0] - 1)
        chunk = ep["actions"][ci].astype(np.float32)
        chunk = self._norm_action(torch.from_numpy(chunk))

        task_name = self._task_name_by_shard[shard_i]
        task_tokens = self.tokens.get(task_name)
        if task_tokens is None:
            task_tokens = self._fallback_cache.get(task_name)
            if task_tokens is None:
                fallback = f"perform the {task_name.replace('-v3', '').replace('-', ' ')} task"
                task_tokens = self.tokenizer(
                    [fallback],
                    padding="max_length",
                    truncation=True,
                    return_tensors="pt",
                ).input_ids
                self._fallback_cache[task_name] = task_tokens

        i = torch.randint(len(task_tokens), (1,)).item()
        tokens = task_tokens[i]

        return {"img": img, "state": state, "chunk": chunk, "tok": tokens}

    def _norm_action(self, x: torch.Tensor) -> torch.Tensor:
        return (x - self.action_mean) / (self.action_std + 1e-8)

class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0

    def update(self, val: float, n: int = 1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / max(self.count, 1)

In [5]:
def save_checkpoint(path: str, model: nn.Module, optimizer, scaler, epoch: int, step: int, best_loss: float):
    torch.save(
        {
            "epoch": epoch,
            "step": step,
            "best_loss": best_loss,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler": scaler.state_dict(),
            "config": model.cfg,
        },
        path,
    )


def load_checkpoint(path: str, model: nn.Module, optimizer, scaler):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    print(f"Resumed from {path} (epoch={ckpt['epoch']}, step={ckpt['step']})")
    return int(ckpt["epoch"]), int(ckpt["step"]), float(ckpt["best_loss"])


def build_model_cfg(stats: dict[str, torch.Tensor]) -> VLAConfig:
    cfg = VLAConfig(**MODEL_KWARGS)
    cfg.action_mean = stats["action_mean"].tolist()
    cfg.action_std = stats["action_std"].tolist()
    return cfg


def build_optimizer(model: VLA, lr: float, backbone_lr_scale: float, weight_decay: float):
    backbone_params, head_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "backbone" in name:
            backbone_params.append(p)
        else:
            head_params.append(p)

    return torch.optim.AdamW(
        [
            {"params": head_params, "lr": lr},
            {"params": backbone_params, "lr": lr * backbone_lr_scale},
        ],
        weight_decay=weight_decay,
        fused=torch.cuda.is_available()
    )

In [8]:
tmp_cfg = VLAConfig(**MODEL_KWARGS)
dataset = VLAEpisodeDatasetCache(
        data_dir=DATA_DIR,
        tokenizer=SiglipTokenizer.from_pretrained(VLAConfig().siglip_model_id),
        cfg=tmp_cfg,
)

Dataset: 20 episodes, 200,000 samples


In [18]:
if 1:
    val_len = max(1, int(len(dataset) * VAL_SPLIT))
    train_len = len(dataset) - val_len
    train_ds, val_ds = random_split(
        dataset,
        [train_len, val_len],
        generator=torch.Generator().manual_seed(SEED),
    )

    workers = NUM_WORKERS
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
    )

In [ ]:
it = iter(train_loader)
next(it)

In [ ]:
def train(dataset):
    seed_everything(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    cfg = build_model_cfg(
        {
            "action_mean": dataset.action_mean,
            "action_std": dataset.action_std,
        }
    )

    val_len = max(1, int(len(dataset) * VAL_SPLIT))
    train_len = len(dataset) - val_len
    train_ds, val_ds = random_split(
        dataset,
        [train_len, val_len],
        generator=torch.Generator().manual_seed(SEED),
    )

    workers = NUM_WORKERS
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        # num_workers=workers,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS and workers > 0,
        drop_last=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        # num_workers=workers,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS and workers > 0,
    )

    model = VLA(cfg, device=device).to(device)
    if COMPILE_MODEL:
        model = torch.compile(model)

    total, trainable = print_model_counts(model)
    print(f"Total params: {total:,} | Trainable: {trainable:,} | Frozen: {total - trainable:,}")

    optimizer = build_optimizer(
        model,
        lr=LEARNING_RATE,
        backbone_lr_scale=BACKBONE_LR_SCALE,
        weight_decay=WEIGHT_DECAY,
    )

    total_steps = EPOCHS * len(train_loader)
    warmup_steps = WARMUP_EPOCHS * len(train_loader)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    bf16_ok = device.type == "cuda" and torch.cuda.is_bf16_supported()
    amp_dtype = torch.bfloat16 if bf16_ok else torch.float16
    amp_enabled = device.type == "cuda"
    scaler = GradScaler(device=device.type, enabled=(amp_enabled and amp_dtype == torch.float16))
    print(f"AMP dtype: {amp_dtype} | GradScaler enabled: {scaler.is_enabled()}")

    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    latest_path = os.path.join(CHECKPOINT_DIR, "latest.pt")
    best_path = os.path.join(CHECKPOINT_DIR, "best.pt")

    start_epoch = 0
    global_step = 0
    best_val = float("inf")

    if RESUME and os.path.exists(latest_path):
        start_epoch, global_step, best_val = load_checkpoint(
            latest_path, model, optimizer, scaler
        )
        start_epoch += 1

    for epoch in tqdm(range(start_epoch, EPOCHS)):
        model.train()
        epoch_train_meter = AverageMeter()
        log_meter = AverageMeter()
        t0 = time.perf_counter()
        print("Starting new epoch.")
        for batch in train_loader:
            img = batch["img"].to(device, non_blocking=True)
            tok = batch["tok"].to(device, non_blocking=True)
            state = batch["state"].to(device, non_blocking=True)
            chunk = batch["chunk"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            autocast_ctx = (
                torch.autocast(device_type=device.type, dtype=amp_dtype)
                if amp_enabled else nullcontext()
            )
            with torch.autocast(device_type=device.type, dtype=amp_dtype):
                loss = model.loss(img, tok, state, chunk)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            epoch_train_meter.update(loss.item(), img.size(0))
            log_meter.update(loss.item(), img.size(0))
            global_step += 1

            if global_step % LOG_EVERY_STEPS == 0:
                elapsed = max(time.perf_counter() - t0, 1e-6)
                sps = LOG_EVERY_STEPS * BATCH_SIZE / elapsed
                lr_now = optimizer.param_groups[0]["lr"]
                print(
                    f"epoch {epoch:>3d} step {global_step:>7d} "
                    f"loss {log_meter.avg:.4f} lr {lr_now:.2e} {sps:,.0f} samples/s"
                )
                log_meter.reset()
                t0 = time.perf_counter()

        model.eval()
        val_meter = AverageMeter()
        with torch.no_grad():
            for batch in val_loader:
                img = batch["img"].to(device, non_blocking=True)
                tok = batch["tok"].to(device, non_blocking=True)
                state = batch["state"].to(device, non_blocking=True).unsqueeze(1)
                chunk = batch["chunk"].to(device, non_blocking=True)

                autocast_ctx = (
                    torch.autocast(device_type=device.type, dtype=amp_dtype)
                    if amp_enabled else nullcontext()
                )
                with autocast_ctx:
                    val_loss = model.loss(img, tok, state, chunk)
                val_meter.update(val_loss.item(), img.size(0))

        print(f"epoch {epoch:>3d} train_loss {epoch_train_meter.avg:.4f} val_loss {val_meter.avg:.4f}")

        save_checkpoint(latest_path, model, optimizer, scaler, epoch, global_step, best_val)
        if val_meter.avg < best_val:
            best_val = val_meter.avg
            save_checkpoint(best_path, model, optimizer, scaler, epoch, global_step, best_val)
            print(f"new best: {best_val:.4f} -> {best_path}")

    print("Training complete.")

In [ ]:
train(dataset)

In [ ]:
os.makedirs("/content/drive/MyDrive/VLA/mt50/model_v1", exist_ok=True)
shutil.copytree("/content/drive/MyDrive/VLA/mt50/model_v1", DATA_DIR+"/model_v1", dirs_exist_ok=True)
from google.colab import runtime
import time
time.sleep(300)  # wait 5m for everything to finish up
runtime.unassign()